In [4]:
# rag_hybrid_vs_dense_with_caches.py
# RAG (LangChain + Chroma) with:
#   1) Hybrid Retrieval: BM25 (sparse) + Chroma (dense) via EnsembleRetriever
#   2) Dense-only baseline (Chroma) for comparison
#   3) Embedding cache → CacheBackedEmbeddings + LocalFileStore
#   4) LLM answer cache → langchain.llm_cache via SQLiteCache
#
# Usage:
#   pip install -U langchain langchain-openai langchain-community chromadb pypdf python-dotenv pandas
#   export OPENAI_API_KEY=sk-...   # (Windows: setx OPENAI_API_KEY "sk-...")
#   python rag_hybrid_vs_dense_with_caches.py
#
# Optional: put PDFs/TXTs under ./data to index; otherwise sample texts are used.

import os, time
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd

# ------- LangChain packages (new style) -------
#from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.chat_models import ChatOpenAI
from langchain.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader
from langchain_community.retrievers import BM25Retriever
from langchain.text_splitter import RecursiveCharacterTextSplitter
#from langchain.retrievers import EnsembleRetriever
from langchain.retrievers.ensemble import EnsembleRetriever

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ------- Caches -------
import langchain
from langchain.cache import SQLiteCache
from langchain.storage import LocalFileStore
from langchain.embeddings import CacheBackedEmbeddings

load_dotenv()

# ---------------------------
# Config
# ---------------------------
PERSIST_DIR     = "./chroma_db1"                       # Chroma persistence
EMB_CACHE_DIR   = "./emb_cache1"                       # Disk cache for embeddings
LLM_CACHE_PATH  = "./.langchain_llm_cache1sqlite"      # SQLite cache for LLM outputs
DATA_DIR        = "./data"                             # PDFs/TXTs go here
COLLECTION      = "docs1"
EMBED_MODEL     = "text-embedding-3-small"             # cheap & good
CHAT_MODEL      = "gpt-4o-mini"                        # any OpenAI chat model

# Toggle: how many documents to retrieve per retriever
K_DENSE = 4
K_SPARSE = 8
K_HYBRID = 6  # final fused k

# LLM answer cache (affects llm.invoke inside chains too)
Path(LLM_CACHE_PATH).parent.mkdir(parents=True, exist_ok=True)
langchain.llm_cache = SQLiteCache(database_path=LLM_CACHE_PATH)

# ---------------------------
# Helpers
# ---------------------------
def load_docs():
    """Load docs from ./data (PDF/TXT). Fall back to small samples."""
    docs = []
    data_path = Path(DATA_DIR)
    if data_path.exists():
        # pass 1: text-like files (DirectoryLoader + TextLoader)
        loader = DirectoryLoader(
            DATA_DIR,
            glob="**/*",
            loader_cls=TextLoader,
            show_progress=True,
            use_multithreading=True,
        )
        try:
            docs.extend(loader.load())
        except Exception:
            pass
        # pass 2: PDFs (PyPDFLoader)
        for pdf in data_path.rglob("*.pdf"):
            try:
                docs.extend(PyPDFLoader(str(pdf)).load())
            except Exception:
                pass

    if not docs:
        from langchain.schema import Document
        docs = [
            Document(page_content=("LangChain is a framework for developing LLM apps. "
                                   "It integrates vector stores like Chroma and supports RAG pipelines."),
                     metadata={"source": "sample:langchain"}),
            Document(page_content=("Chroma is an open-source embedding DB (vector store) that stores "
                                   "document embeddings and enables similarity search."),
                     metadata={"source": "sample:chroma"}),
        ]
    return docs


def build_or_load_chroma(cached_embeddings) -> Chroma:
    """Create/load a persistent Chroma index using cached embeddings."""
    Path(PERSIST_DIR).mkdir(parents=True, exist_ok=True)
    index_exists = (Path(PERSIST_DIR) / "chroma.sqlite3").exists()
    if index_exists:
        return Chroma(
            collection_name=COLLECTION,
            embedding_function=cached_embeddings,
            persist_directory=PERSIST_DIR,
        )

    docs = load_docs()
    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
    chunks = splitter.split_documents(docs)

    vs = Chroma.from_documents(
        documents=chunks,
        embedding=cached_embeddings,
        collection_name=COLLECTION,
        persist_directory=PERSIST_DIR,
    )
    vs.persist()
    return vs


def build_bm25_from_vstore_text(vstore: Chroma) -> BM25Retriever:
    """
    Build a BM25 retriever using the same chunked documents stored in Chroma.
    Chroma can return all texts; we rehydrate documents for BM25 indexing.
    """
    # NOTE: Chroma doesn't expose "all docs" directly; to keep it simple and fast:
    # we re-split from the original docs just like we did for Chroma (keeps parity).
    docs = load_docs()
    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
    chunks = splitter.split_documents(docs)

    bm25 = BM25Retriever.from_documents(chunks)
    bm25.k = K_SPARSE
    return bm25


def make_prompt():
    return ChatPromptTemplate.from_messages(
        [
            ("system",
             "You are a concise, helpful assistant. Use the provided context to answer.\n"
             "If the answer isn't in the context, say you don't know.\n\n"
             "Cite sources inline at the end as [source:<short>].\n\n"
             "Context:\n{context}"),
            ("human", "{question}"),
        ]
    )


def format_docs(docs):
    return "\n\n".join(
        f"Source: {d.metadata.get('source','?')}\n{d.page_content}" for d in docs
    )


def make_rag_chain(retriever, llm):
    """RAG chain: retrieve → prompt → LLM → string."""
    prompt = make_prompt()
    chain = (
        {
            "context": retriever | (lambda docs: format_docs(docs)),
            "question": RunnablePassthrough(),
        }
        | prompt
        | llm
        | StrOutputParser()
    )
    return chain


def timed(fn):
    def _inner(*args, **kwargs):
        t0 = time.time()
        out = fn(*args, **kwargs)
        return out, time.time() - t0
    return _inner


def sources_short(docs):
    """Compact semicolon-joined sources list for logging."""
    return ";".join([str(d.metadata.get("source","?")) for d in docs])


# ---------------------------
# Main
# ---------------------------
if __name__ == "__main__":
    # 1) Embedding cache: stores content-hash → vector on disk
    Path(EMB_CACHE_DIR).mkdir(parents=True, exist_ok=True)
    base_embeddings = OpenAIEmbeddings(model=EMBED_MODEL)
    byte_store = LocalFileStore(EMB_CACHE_DIR)
    cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
        base_embeddings,
        byte_store,
        namespace=f"{EMBED_MODEL}-v1",   # separate namespace per settings/model
    )

    # 2) Persistent Chroma index
    vstore = build_or_load_chroma(cached_embeddings)

    # Dense-only retriever (baseline)
    dense_retriever = vstore.as_retriever(search_kwargs={"k": K_DENSE})

    # Sparse retriever (BM25)
    bm25_retriever = build_bm25_from_vstore_text(vstore)

    # Hybrid (Ensemble) retriever
    # Weights roughly balance sparse vs dense; tweak as needed.
    hybrid_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, dense_retriever],
        weights=[0.55, 0.45],  # slightly favor sparse for exact lexical matches
        # NOTE: EnsembleRetriever uses normalized scores & RRF-like fusion internally.
        # It will return up to 'k' merged results:
        search_kwargs={"k": K_HYBRID},
    )

    # 3) LLM (benefits from SQLite LLM cache set above)
    llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)

    # 4) RAG chains
    chain_dense = make_rag_chain(dense_retriever, llm)
    chain_hybrid = make_rag_chain(hybrid_retriever, llm)

    # 5) Evaluation input CSV
    #    Expect a "Question" column (same as your original script)
    #    Adjust to your path as needed.
    input_csv = r"C:\Users\surya.adatravu\Documents\RAGAnalysis\RA_FSM_QA.csv"
    df = pd.read_csv(input_csv)

    # Output columns
    df["t_dense"] = 0.0
    df["t_hybrid"] = 0.0
    df["ans_dense"] = ""
    df["ans_hybrid"] = ""
    df["src_dense"] = ""
    df["src_hybrid"] = ""

    # We'll also store top-k docs used for each mode for comparison.
    # We do this by asking each retriever directly (without the LLM) once per question.
    for i in range(df.shape[0]):
        question = str(df.loc[i, "Question"])

        # Retrieve top docs (for source logging)
        dense_docs = dense_retriever.get_relevant_documents(question)
        hybrid_docs = hybrid_retriever.get_relevant_documents(question)

        # Dense answer
        (ans_d), t_d = timed(chain_dense.invoke)(question)
        df.loc[i, "t_dense"] = round(t_d, 3)
        df.loc[i, "ans_dense"] = ans_d
        df.loc[i, "src_dense"] = sources_short(dense_docs)

        # Hybrid answer
        (ans_h), t_h = timed(chain_hybrid.invoke)(question)
        df.loc[i, "t_hybrid"] = round(t_h, 3)
        df.loc[i, "ans_hybrid"] = ans_h
        df.loc[i, "src_hybrid"] = sources_short(hybrid_docs)

    # 6) Save comparison
    out_csv = "results_hybrid_vs_dense.csv"
    df.to_csv(out_csv, index=False)

    # 7) Report cache locations
    print("\nDone. Comparison written to:", Path(out_csv).resolve())
    print("\nCache locations:")
    print(f"• Chroma DB:        {Path(PERSIST_DIR).resolve()}")
    print(f"• Embedding cache:  {Path(EMB_CACHE_DIR).resolve()}")
    print(f"• LLM cache (SQL):  {Path(LLM_CACHE_PATH).resolve()}")


100%|███████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 207.64it/s]
C:\Users\surya.adatravu\AppData\Local\anaconda3\Lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 0.2.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import ChatOpenAI`.
  warn_deprecated(
C:\Users\surya.adatravu\AppData\Local\anaconda3\Lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(



Done. Comparison written to: C:\Users\surya.adatravu\Documents\RAG_HRETREIVER\results_hybrid_vs_dense.csv

Cache locations:
• Chroma DB:        C:\Users\surya.adatravu\Documents\RAG_HRETREIVER\chroma_db1
• Embedding cache:  C:\Users\surya.adatravu\Documents\RAG_HRETREIVER\emb_cache1
• LLM cache (SQL):  C:\Users\surya.adatravu\Documents\RAG_HRETREIVER\.langchain_llm_cache1sqlite
